# 🚀 Google Colab Live AI Video API Server (RAM Optimized + ngrok First)
Bu notebook, **RAM çökmesini önleyen CPU Offload teknolojisi** ile çalışır. Önce ngrok canlı bağlantı adresini verir, ardından modeli GPU'ya yükler.

In [ ]:
# 1. Gerekli Kütüphanelerin Kurulumu
!pip install -q diffusers transformers accelerate torch torchvision imageio-ffmpeg fastapi uvicorn pyngrok nest_asyncio hf_transfer
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
print('✅ Tüm kütüphaneler kuruldu!')

In [ ]:
# 2. LTX-Video Modelini Düşük RAM Modunda Yükleme (Çökme Engelleyici)
import torch
from diffusers import LTXPipeline
from diffusers.utils import export_to_video

print('🚀 AI Video Modeli Yükleniyor (RAM Korumalı)...')
pipe = LTXPipeline.from_pretrained(
    'Lightricks/LTX-Video',
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True
)
pipe.enable_model_cpu_offload()
print('✅ AI Video Modeli RAM Çökmesi Olmadan Başarıyla Yüklendi!')

In [ ]:
# 3. Canlı FastAPI + ngrok Web Sunucusu
from fastapi import FastAPI, Response
from pydantic import BaseModel
import uvicorn
import nest_asyncio
from pyngrok import ngrok

app = FastAPI()

class VideoRequest(BaseModel):
    prompt: str
    niche: str = 'minecraft'
    width: int = 576
    height: int = 1024

@app.get('/')
def health_check():
    return {'status': 'online', 'model': 'LTX-Video'}

@app.post('/generate_video')
def generate_video(req: VideoRequest):
    print(f'🎬 Otomasyondan video isteği alındı: {req.prompt}')
    frames = pipe(
        prompt=req.prompt,
        negative_prompt='low quality, blurry, distorted',
        width=req.width,
        height=req.height,
        num_frames=121,
        num_inference_steps=25
    ).frames[0]
    
    out_path = '/content/colab_generated_video.mp4'
    export_to_video(frames, out_path, fps=24)
    
    with open(out_path, 'rb') as f:
        return Response(content=f.read(), media_type='video/mp4')

public_url = ngrok.connect(8000)
print('====================================================')
print('🚀 CANLI GOOGLE COLAB API URL ADRESİNİZ:')
print(public_url)
print('====================================================')
print(f'COLAB_API_URL={public_url}')
print('====================================================')

nest_asyncio.apply()
uvicorn.run(app, host='0.0.0.0', port=8000)